# Issue #2 — Pipeline e Modelagem
Desenvolvimento do pipeline de Machine Learning para previsão de inadimplência.

# Imports das bibliotecas

In [2]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Carregando o dataset (arquivo .csv)

In [7]:
CAMINHO_DADOS = Path("..") / "data" / "credito.csv"

df = pd.read_csv(CAMINHO_DADOS)

df.head()

,id_contrato,idade,renda_mensal,tempo_emprego_anos,score_credito,dividas_ativas,possui_imovel,finalidade,prazo_meses,valor_emprestimo,inadimplente
0,E00001,44.0,4810.0,3.3,645.0,0,sim,negocio,60,7800.0,0
1,E00002,28.0,4070.0,1.1,805.0,3,nao,pessoal,24,7100.0,0
2,E00003,49.0,7050.0,5.2,563.0,1,sim,veiculo,48,29000.0,0
3,E00004,51.0,6040.0,0.8,778.0,1,sim,pessoal,24,21400.0,0
4,E00005,18.0,1300.0,2.4,551.0,0,nao,negocio,24,4700.0,1


# Separação entre features e target

Utilizando a variável `inadimplente` como target (`y`).
Removendo a coluna `id_contrato` das features, pois representa apenas um identificador único do contrato e não uma característica útil para a previsão.

In [8]:
X = df.drop(columns=["inadimplente", "id_contrato"])
y = df["inadimplente"]

print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

X.head()

Dimensão de X: (6000, 9)
Dimensão de y: (6000,)


,idade,renda_mensal,tempo_emprego_anos,score_credito,dividas_ativas,possui_imovel,finalidade,prazo_meses,valor_emprestimo
0,44.0,4810.0,3.3,645.0,0,sim,negocio,60,7800.0
1,28.0,4070.0,1.1,805.0,3,nao,pessoal,24,7100.0
2,49.0,7050.0,5.2,563.0,1,sim,veiculo,48,29000.0
3,51.0,6040.0,0.8,778.0,1,sim,pessoal,24,21400.0
4,18.0,1300.0,2.4,551.0,0,nao,negocio,24,4700.0


# Divisão entre treino e teste

O dataset será dividido em 80% para treinamento e 20% para teste (conforme o material do Professor).
A divisão será estratificada pela variável alvo para preservar a proporção de adimplentes e inadimplentes nos dois conjuntos.

In [9]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X treino:", X_treino.shape)
print("X teste:", X_teste.shape)
print("y treino:", y_treino.shape)
print("y teste:", y_teste.shape)



print("Distribuição original:")
print(y.value_counts(normalize=True).round(4))

print("\nDistribuição treino:")
print(y_treino.value_counts(normalize=True).round(4))

print("\nDistribuição teste:")
print(y_teste.value_counts(normalize=True).round(4))

X treino: (4800, 9)
X teste: (1200, 9)
y treino: (4800,)
y teste: (1200,)
Distribuição original:
inadimplente
0    0.7963
1    0.2037
Name: proportion, dtype: float64

Distribuição treino:
inadimplente
0    0.7962
1    0.2037
Name: proportion, dtype: float64

Distribuição teste:
inadimplente
0    0.7967
1    0.2033
Name: proportion, dtype: float64


# Identificação das features

As variáveis serão separadas entre numéricas e categóricas para que cada grupo receba o pré-processamento adequado antes do treinamento dos modelos.

In [10]:
features_numericas = [
    "idade",
    "renda_mensal",
    "tempo_emprego_anos",
    "score_credito",
    "dividas_ativas",
    "prazo_meses",
    "valor_emprestimo",
]

features_categoricas = [
    "possui_imovel",
    "finalidade",
]


print("Features numéricas:")
print(features_numericas)

print("\nFeatures categóricas:")
print(features_categoricas)

print("\nTotal:", len(features_numericas) + len(features_categoricas))

Features numéricas:
['idade', 'renda_mensal', 'tempo_emprego_anos', 'score_credito', 'dividas_ativas', 'prazo_meses', 'valor_emprestimo']

Features categóricas:
['possui_imovel', 'finalidade']

Total: 9


# Construção do pré-processamento

Para as variáveis numéricas, os valores ausentes serão preenchidos utilizando a mediana e posteriormente será aplicada padronização com `StandardScaler`.

In [11]:
pipeline_numerico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# União dos pré-processamentos

O `ColumnTransformer` é utilizado para aplicar tratamentos diferentes às variáveis numéricas e categóricas dentro de uma única etapa de pré-processamento.

In [12]:
preprocessador = ColumnTransformer(
    transformers=[
        ("numericas", pipeline_numerico, features_numericas),
        ("categoricas", pipeline_categorico, features_categoricas),
    ]
)

# Definição dos modelos

Algoritmos que serão avaliados:

- Regressão Logística
- Árvore de Decisão
- Random Forest

Todos os modelos utilizarão o mesmo pré-processamento para permitir uma comparação consistente entre os algoritmos.

In [13]:
modelos = {
    "Regressão Logística": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Árvore de Decisão": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),
}

# Validação cruzada

Os modelos serão comparados utilizando validação cruzada estratificada com 5 folds.

A métrica utilizada será ROC AUC, permitindo avaliar a capacidade dos modelos de distinguir contratos inadimplentes de não inadimplentes.

O conjunto de teste não será utilizado durante essa comparação.

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Comparação dos modelos

Para cada algoritmo será criado um Pipeline contendo o pré-processamento e o modelo.

A avaliação será realizada por validação cruzada utilizando ROC AUC.

In [14]:
resultados = []

for nome, modelo in modelos.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessador", preprocessador),
            ("modelo", modelo),
        ]
    )

    scores = cross_val_score(
        pipeline,
        X_treino,
        y_treino,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    resultados.append({
        "Modelo": nome,
        "ROC AUC médio": scores.mean(),
        "Desvio padrão": scores.std(),
    })


resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(
    by="ROC AUC médio",
    ascending=False
).reset_index(drop=True)

resultados_df

,Modelo,ROC AUC médio,Desvio padrão
0,Regressão Logística,0.761253,0.009790
1,Random Forest,0.730015,0.009994
2,Árvore de Decisão,0.575991,0.015674
